# Example 11.4.3 / 2: Bounds for Global Optimization

Companion notebook for **11.4.3 Local and Global Nonlinear Solvers** in the AMPL lab Documentation, section "Bounds for Global Optimization". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Bounds for Global Optimization

Spatial branch-and-bound, the method of deterministic global solvers, splits variable ranges into
regions, builds a convex relaxation over each region and compares its lower bound with the best
feasible point found. Tight, valid variable bounds strengthen those relaxations; missing bounds may
prevent a proof or make it prohibitively slow. Bounds must come from the modeled system, not be
chosen merely to force the solver to finish.

Couenne's log shows the role of bounds. With $-2\le x\le2$ declared it closes the gap with no
branching. With no bounds it first derives its own (the `improved bounds` lines) and then reaches
the same bound. A polynomial this small admits such derived bounds; a model whose expressions give
the solver nothing to bound may not. The run captures the log with `ampl.get_output("solve;")`
and keeps the lines about bounds and nodes:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param lo default -Infinity;  # lower bound on x
param hi default Infinity;   # upper bound on x

var x >= lo, <= hi;                     # bounds set from data
minimize Wells: (x^2 - 1)^2 + 0.1 * x;  # nonconvex: two local minima

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.option["solver"] = "couenne"
for lo, hi in ((-2, 2), (float("-inf"), float("inf"))):
    ampl.param["lo"] = lo
    ampl.param["hi"] = hi
    print("bounds", lo, hi)
    for line in ampl.get_output("solve;").splitlines():
        if "improved bounds" in line or line.startswith(("Lower bound", "Upper bound", "Branch-and-bound")):
            print("  ", " ".join(line.split()))

bounds -2 2
   Lower bound: -0.100617
   Upper bound: -0.100617 (gap: 0.00%)
   Branch-and-bound nodes: 0
bounds -inf inf
   Optimality Based BT: 2 improved bounds
   Probing: 4 improved bounds
   Lower bound: -0.100617
   Upper bound: -0.100617 (gap: 0.00%)
   Branch-and-bound nodes: 0
